<a href="https://colab.research.google.com/github/gisele-mgs/especializacacin/blob/main/DL_meme_analisis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision transformers timm

### BERT (texto)

É uma rede neural profunda baseada em Transformers.

Possui dezenas de camadas de atenção e feed-forward (12 camadas na versão base, 768 hidden units).

Aprende representações contextuais profundas das palavras.

### EfficientNet (imagem)

É uma rede convolucional profunda (CNN) pré-treinada no ImageNet.

Possui dezenas de camadas convolucionais e pooling para extrair features de imagens.

### MLP de classificação (camadas densas)

É uma rede totalmente conectada adicionada no topo para fazer a fusão dos embeddings.

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel
import timm  # EfficientNet e outros modelos de visão

In [ ]:
# ------------------------
# Modelo Multimodal
# ------------------------
class MultimodalClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(MultimodalClassifier, self).__init__()

        # Text encoder (BERT)
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.text_dim = 768

        # Image encoder (EfficientNet-B0)
        self.effnet = timm.create_model("efficientnet_b0", pretrained=True)
        self.effnet.classifier = nn.Identity()  # remove última camada de classificação
        self.img_dim = self.effnet.num_features  # 1280 para B0

        # Fusion + classifier
        fusion_dim = self.text_dim + self.img_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            # Pode experimentar com uma taxa de dropout maior aqui, por exemplo, 0.4 ou 0.5
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, images):
        # Texto → BERT
        text_outputs = self.bert(input_ids=input_ids,
                                 attention_mask=attention_mask)
        text_emb = text_outputs.pooler_output  # [batch, 768]

        # Imagem → EfficientNet
        img_emb = self.effnet(images)  # [batch, 1280]

        # Fusão
        fusion = torch.cat((text_emb, img_emb), dim=1)

        # Classificação
        logits = self.classifier(fusion)
        return logits

# OPÇÃO COM EFFICIENT NET
<!-- class MultimodalClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(MultimodalClassifier, self).__init__()

        # Text encoder (BERT)
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.text_dim = 768

        # Image encoder (ResNet50)
        # Replacing EfficientNet with ResNet50
        self.resnet = timm.create_model("resnet50", pretrained=True)
        # Remove the final classification layer to get features
        self.resnet.fc = nn.Identity()
        self.img_dim = self.resnet.num_features # Typically 2048 for ResNet50


        # Fusion + classifier
        fusion_dim = self.text_dim + self.img_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            # Pode experimentar com uma taxa de dropout maior aqui, por exemplo, 0.4 ou 0.5
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, images):
        # Texto → BERT
        text_outputs = self.bert(input_ids=input_ids,
                                 attention_mask=attention_mask)
        text_emb = text_outputs.pooler_output  # [batch, 768]

        # Imagem → ResNet
        # Using the ResNet model for image encoding
        img_emb = self.resnet(images)  # [batch, 2048]


        # Fusão
        fusion = torch.cat((text_emb, img_emb), dim=1)

        # Classificação
        logits = self.classifier(fusion)
        return logits -->

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode_texts(texts, max_len=64):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )


In [ ]:
# from torchvision import transforms
# from PIL import Image

# image_transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],  # ImageNet stats
#         std=[0.229, 0.224, 0.225]
#     )
# ])
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor()
])

In [ ]:
# Dados de exemplo (mini batch)
texts = ["This is a nice meme", "This is an offensive meme"]
images = [Image.open("/content/drive/MyDrive/Colab Notebooks/Deep_Learning/Multimodal-Meme-Classification-Identifying-Offensive-Content-in-Image-and-Text/MultiOFF_Dataset/Labelled Images/00DjNzR.png"),
          Image.open("/content/drive/MyDrive/Colab Notebooks/Deep_Learning/Multimodal-Meme-Classification-Identifying-Offensive-Content-in-Image-and-Text/MultiOFF_Dataset/Labelled Images/05nBEvh.png")]

# Texto
encodings = encode_texts(texts)
input_ids, attention_mask = encodings["input_ids"], encodings["attention_mask"]

# Imagem / antes image_transform
images_tensor = torch.stack([train_transform(img) for img in images])

# Modelo
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MultimodalClassifier(num_classes=2).to(device)

# Forward pass
with torch.no_grad():
    logits = model(input_ids.to(device),
                   attention_mask.to(device),
                   images_tensor.to(device))
print("Logits:", logits)
print("Pred:", torch.argmax(logits, dim=1))


Logits: tensor([[ 0.3066, -0.0091],
        [-0.0344, -0.0324]], device='cuda:0')
Pred: tensor([0, 1], device='cuda:0')


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import os # Import the os module
from PIL import Image # Import Image here as well

# Exemplo de Dataset multimodal
class MemeDataset(Dataset):
    def __init__(self, texts, img_paths_dir, image_names, labels): # Modified to accept image_names and img_paths_dir
        self.texts = texts
        self.img_paths_dir = img_paths_dir # Store the directory path
        self.image_names = image_names # Store the list of image names
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        img_name = self.image_names[idx] # Get the image name
        img_path = os.path.join(self.img_paths_dir, img_name) # Construct the full image path

        try:
            img = Image.open(img_path).convert("RGB") # Use the full path
            img_tensor = image_transform(img)
        except FileNotFoundError:
            print(f"Warning: Image not found at {img_path}. Skipping this sample.")
            return None # Return None if image is not found

        label = self.labels[idx]

        enc = encode_texts([text])
        input_ids = enc["input_ids"].squeeze(0)
        att_mask = enc["attention_mask"].squeeze(0)


        return input_ids, att_mask, img_tensor, torch.tensor(label)

In [ ]:
import pandas as pd
import os

train_csv_path = "/content/drive/MyDrive/Colab Notebooks/Deep_Learning/Multimodal-Meme-Classification-Identifying-Offensive-Content-in-Image-and-Text/MultiOFF_Dataset/Split Dataset/Training_meme_dataset.csv"
val_csv_path = "/content/drive/MyDrive/Colab Notebooks/Deep_Learning/Multimodal-Meme-Classification-Identifying-Offensive-Content-in-Image-and-Text/MultiOFF_Dataset/Split Dataset/Validation_meme_dataset.csv"
test_csv_path = "/content/drive/MyDrive/Colab Notebooks/Deep_Learning/Multimodal-Meme-Classification-Identifying-Offensive-Content-in-Image-and-Text/MultiOFF_Dataset/Split Dataset/Testing_meme_dataset.csv"
img_dir_path = "/content/drive/MyDrive/Colab Notebooks/Deep_Learning/Multimodal-Meme-Classification-Identifying-Offensive-Content-in-Image-and-Text/MultiOFF_Dataset/Labelled Images"

# Função para carregar e processar os dados
def load_and_process_data(csv_path, img_dir_path):
    df = pd.read_csv(csv_path)
    existing_images = [f for f in os.listdir(img_dir_path) if os.path.isfile(os.path.join(img_dir_path, f))]
    df_filtered = df[df['image_name'].isin(existing_images)].copy() # Usar .copy() para evitar SettingWithCopyWarning

    # Converter rótulos de string para numéricos
    label_map = {'Non-offensiv': 0, 'offensive': 1}
    labels = [label_map[label] for label in df_filtered['label'].tolist()]
    texts = df_filtered['sentence'].tolist()
    imgs_names = df_filtered['image_name'].to_list()

    return texts, imgs_names, labels, len(df), len(df_filtered)

# Carregar e processar dados de treinamento
train_texts, train_imgs_names, train_labels, original_train_size, filtered_train_size = load_and_process_data(train_csv_path, img_dir_path)
print(f"Tamanho original do dataset de treinamento: {original_train_size}")
print(f"Tamanho filtrado do dataset de treinamento (imagens existentes): {filtered_train_size}")

# Carregar e processar dados de validação
val_texts, val_imgs_names, val_labels, original_val_size, filtered_val_size = load_and_process_data(val_csv_path, img_dir_path)
print(f"Tamanho original do dataset de validação: {original_val_size}")
print(f"Tamanho filtrado do dataset de validação (imagens existentes): {filtered_val_size}")

# Carregar e processar dados de teste
test_texts, test_imgs_names, test_labels, original_test_size, filtered_test_size = load_and_process_data(test_csv_path, img_dir_path)
print(f"Tamanho original do dataset de teste: {original_test_size}")
print(f"Tamanho filtrado do dataset de teste (imagens existentes): {filtered_test_size}")

Tamanho original do dataset de treinamento: 445
Tamanho filtrado do dataset de treinamento (imagens existentes): 444
Tamanho original do dataset de validação: 149
Tamanho filtrado do dataset de validação (imagens existentes): 148
Tamanho original do dataset de teste: 149
Tamanho filtrado do dataset de teste (imagens existentes): 148


In [ ]:
def collate_fn(batch):
    # Filtrar valores None
    batch = [item for item in batch if item is not None]
    if not batch:
        return None # Retornar None se o lote inteiro for None

    input_ids = torch.stack([item[0] for item in batch])
    att_mask = torch.stack([item[1] for item in batch])
    imgs = torch.stack([item[2] for item in batch])
    labels = torch.stack([item[3] for item in batch])

    return input_ids, att_mask, imgs, labels

In [ ]:
# Create Validation Dataset and DataLoader
val_dataset = MemeDataset(val_texts, img_dir_path, val_imgs_names, val_labels)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

print(f"DataLoader de Validação criado com {len(val_dataset)} amostras.")

DataLoader de Validação criado com 148 amostras.


In [ ]:
# Create Test Dataset and DataLoader
test_dataset = MemeDataset(test_texts, img_dir_path, test_imgs_names, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

print(f"DataLoader de Teste criado com {len(test_dataset)} amostras.")

DataLoader de Teste criado com 148 amostras.


In [ ]:
# Evaluation Function
def evaluate_model(model, dataloader, criterion, device):
    model.eval() # Set model to evaluation mode
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation
        for batch in dataloader:
            if batch is None: # Skip batch if collate_fn returned None
                continue

            input_ids, att_mask, imgs, labels = batch
            input_ids, att_mask = input_ids.to(device), att_mask.to(device)
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(input_ids, att_mask, imgs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0
    accuracy = correct_predictions / total_samples if total_samples > 0 else 0

    return avg_loss, accuracy

Suponha que você já tem listas texts, img_paths, labels
Precisamos passar os nomes das imagens do dataframe para o dataset
Pode experimentar com diferentes taxas de aprendizado, por exemplo, 1e-5, 5e-5, etc.
Adicionando weight_decay para L2 regularization (pode ajustar o valor)

In [ ]:
# Listas dos dados
train_dataset = MemeDataset(train_texts, img_dir_path, train_imgs_names, train_labels)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

In [ ]:
# 🔹 Congelar BERT e EfficientNet no início
for param in model.bert.parameters():
    param.requires_grad = False

for param in model.effnet.parameters():
    param.requires_grad = False


In [ ]:
# Otimizador e Loss

optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:

# Scheduler de taxa de aprendizado (opcional, ajuda a ajustar a taxa de aprendizado durante o treino)
# import torch.optim.lr_scheduler as lr_scheduler # Já importado acima

# Exemplo: StepLR - diminui a taxa de aprendizado a cada 'step_size' épocas por um fator 'gamma'
# scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)


In [ ]:
# Loop de treino com validação
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    correct_train_predictions = 0
    total_train_samples = 0

    for batch in train_loader:
        if batch is None: # Pular lote se collate_fn retornou None
            continue

        input_ids, att_mask, imgs, labels = batch
        input_ids, att_mask = input_ids.to(device), att_mask.to(device)
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, att_mask, imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        # Calcular acurácia de treinamento
        _, predicted = torch.max(outputs.data, 1)
        total_train_samples += labels.size(0)
        correct_train_predictions += (predicted == labels).sum().item()

    avg_train_loss = total_train_loss / len(train_loader) if len(train_loader) > 0 else 0
    train_accuracy = correct_train_predictions / total_train_samples if total_train_samples > 0 else 0

    print(f"Época {epoch+1}/{num_epochs}, Loss Treinamento: {avg_train_loss:.4f}, Acurácia Treinamento: {train_accuracy:.4f}")

    # Etapa de Validação
    val_loss, val_accuracy = evaluate_model(model, val_loader, criterion, device)
    print(f"Época {epoch+1}/{num_epochs}, Loss Validação: {val_loss:.4f}, Acurácia Validação: {val_accuracy:.4f}")

    # Passo do scheduler (se estiver usando um)
    # scheduler.step()

# Etapa de Teste Final
print("\nAvaliação no conjunto de Teste:")
test_loss, test_accuracy = evaluate_model(model, test_loader, criterion, device)
print(f"Loss Teste: {test_loss:.4f}, Acurácia Teste: {test_accuracy:.4f}")

Época 1/10, Loss Treinamento: 0.6258, Acurácia Treinamento: 0.6712
Época 1/10, Loss Validação: 0.6618, Acurácia Validação: 0.5946
Época 2/10, Loss Treinamento: 0.6178, Acurácia Treinamento: 0.6689
Época 2/10, Loss Validação: 0.6634, Acurácia Validação: 0.6081
Época 3/10, Loss Treinamento: 0.6084, Acurácia Treinamento: 0.7027
Época 3/10, Loss Validação: 0.6684, Acurácia Validação: 0.5946
Época 4/10, Loss Treinamento: 0.6107, Acurácia Treinamento: 0.6599
Época 4/10, Loss Validação: 0.6681, Acurácia Validação: 0.5946
Época 5/10, Loss Treinamento: 0.6033, Acurácia Treinamento: 0.6757
Época 5/10, Loss Validação: 0.6786, Acurácia Validação: 0.5946
Época 6/10, Loss Treinamento: 0.5987, Acurácia Treinamento: 0.6802
Época 6/10, Loss Validação: 0.6733, Acurácia Validação: 0.6014
Época 7/10, Loss Treinamento: 0.5957, Acurácia Treinamento: 0.6847
Época 7/10, Loss Validação: 0.6703, Acurácia Validação: 0.6149
Época 8/10, Loss Treinamento: 0.5830, Acurácia Treinamento: 0.6892
Época 8/10, Loss Valida